# Notebook 35 — MEPI-assisted frequency screening

Reproduce only the manuscript frequency-screening demonstration from the final frozen model selected in notebook 34. This is not a new optimization study and must not tune on test evidence.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks": PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
from src.mepi_v1.final_model_manifest_v1_5 import sha256_file, validate_final_model_manifest
from src.mepi_v1.frequency_screening_v1_5 import (
    audit_frequency_screening_inputs,
    run_frequency_screening,
)

RUN_FREQUENCY_SCREENING = False
FINAL_MODEL_MANIFEST = PROJECT_ROOT / "experiments/finetune_v1_5_xlstm_depth8/final_model_manifest.json"
FINAL_MODEL_READY = False
if not FINAL_MODEL_MANIFEST.is_file():
    raise FileNotFoundError(f"Required frozen final-model manifest is missing: {FINAL_MODEL_MANIFEST}")
manifest = validate_final_model_manifest(PROJECT_ROOT, FINAL_MODEL_MANIFEST)

if manifest["final_config_frozen"] is not True:
    raise RuntimeError("Final configuration is not frozen")
if manifest["final_test_complete"] is not True:
    raise RuntimeError("Exactly-once final test is not complete")
if manifest["test_evaluation_count"] != 1:
    raise RuntimeError("Final test evaluation count must be exactly one")
if manifest["configuration_id"] != "l3_1_l4_0p05":
    raise RuntimeError("Frozen selected configuration changed")
if manifest["lambda3"] != 1.0 or manifest["lambda4"] != 0.05:
    raise RuntimeError("Frozen selected lambda values changed")
checkpoint_path = PROJECT_ROOT / manifest["checkpoint_path"]
if not checkpoint_path.is_file():
    raise FileNotFoundError(f"Frozen selected checkpoint is missing: {checkpoint_path}")
if sha256_file(checkpoint_path) != manifest["checkpoint_sha256"]:
    raise RuntimeError("Frozen selected-checkpoint SHA256 mismatch")

FINAL_CONFIG_FROZEN = manifest["final_config_frozen"]
FINAL_TEST_COMPLETE = manifest["final_test_complete"]
TEST_EVALUATION_COUNT = manifest["test_evaluation_count"]
preflight = audit_frequency_screening_inputs(PROJECT_ROOT)
if preflight["status"] != "PASS":
    raise RuntimeError("Frequency-screening preflight did not pass")
if preflight["checkpoint_sha256"] != manifest["checkpoint_sha256"]:
    raise RuntimeError("Preflight checkpoint SHA256 does not match the frozen manifest")
if preflight["screening_condition_count"] + preflight["excluded_hard_fail_count"] != 150:
    raise RuntimeError("Dedicated demo manifest must contain exactly 150 source rows")
if preflight["screening_condition_count"] != 146 or preflight["excluded_hard_fail_count"] != 4:
    raise RuntimeError("Expected 146 usable rows and four VOUT_THD hard-fail exclusions")
if preflight["candidate_frequency_count"] != 15:
    raise RuntimeError("Expected the frozen 15-frequency candidate grid")

FINAL_MODEL_READY = manifest["frequency_screening_ready"] is True and preflight["FINAL_MODEL_READY"] is True
FREQUENCY_SCREENING_INPUT_READY = preflight["FREQUENCY_SCREENING_INPUT_READY"] is True
FREQUENCY_SCREENING_RUNTIME_READY = preflight["FREQUENCY_SCREENING_RUNTIME_READY"] is True
if RUN_FREQUENCY_SCREENING and not (FINAL_MODEL_READY and FREQUENCY_SCREENING_INPUT_READY):
    raise RuntimeError("Notebook 35 requires the frozen final model and dedicated demo input")

if RUN_FREQUENCY_SCREENING:
    screening_result = run_frequency_screening(PROJECT_ROOT, authorize=True)
else:
    screening_result = preflight

print("SELECTED_CONFIGURATION =", manifest["configuration_id"])
print("SELECTED_CHECKPOINT =", manifest["checkpoint_path"])
print("SELECTED_CHECKPOINT_SHA256_MATCH = TRUE")
print("FINAL_MODEL_READY =", str(FINAL_MODEL_READY).upper())
print("FREQUENCY_SCREENING_INPUT_READY =", str(FREQUENCY_SCREENING_INPUT_READY).upper())
print("FREQUENCY_SCREENING_RUNTIME_READY =", str(FREQUENCY_SCREENING_RUNTIME_READY).upper())
print("FREQUENCY_SCREENING_RUN =", str(RUN_FREQUENCY_SCREENING).upper())
print("SCREENING_INPUT_SHA256 =", preflight["screening_input_sha256"])
print("SCREENING_WAVEFORM_BUNDLE_SHA256 =", preflight["screening_waveform_bundle_sha256"])
print("SCREENING_SOURCE_ROWS =", preflight["screening_condition_count"] + preflight["excluded_hard_fail_count"])
print("SCREENING_USABLE_ROWS =", preflight["screening_condition_count"])
print("SCREENING_EXCLUDED_HARD_FAIL_ROWS =", preflight["excluded_hard_fail_count"])
print("SCREENING_CANDIDATE_FREQUENCIES =", preflight["candidate_frequency_count"])
print("FINAL_CONFIG_FROZEN = TRUE")
print("FINAL_TEST_COMPLETE = TRUE")
print("TRAINING_PERFORMED = FALSE")
print("SCIENTIFIC_TRAINING_RUN = FALSE")
print("TEST_DATA_USED_FOR_SCREENING = FALSE")
print("FINAL_TEST_RERUN = FALSE")
print("TEST_EVALUATION_COUNT =", TEST_EVALUATION_COUNT)
print("TEST_TUNING = FALSE")


SELECTED_CONFIGURATION = l3_1_l4_0p05
SELECTED_CHECKPOINT = experiments/finetune_v1_5_xlstm_depth8/loss_weight_sensitivity/l3_1_l4_0p05/best_checkpoint.pt
SELECTED_CHECKPOINT_SHA256_MATCH = TRUE
FINAL_MODEL_READY = TRUE
FREQUENCY_SCREENING_INPUT_READY = TRUE
FREQUENCY_SCREENING_RUNTIME_READY = TRUE
FREQUENCY_SCREENING_RUN = TRUE
SCREENING_INPUT_SHA256 = 812ea7e87ad69c02702a411d0f1fe6ef241b9753586a50f1d38522d25d7f9556
SCREENING_WAVEFORM_BUNDLE_SHA256 = ce27eb62f45a6282ae8b49ceb7c9afd17082f7cf1eb04d8fe9683ebcc6d807d6
SCREENING_SOURCE_ROWS = 150
SCREENING_USABLE_ROWS = 146
SCREENING_EXCLUDED_HARD_FAIL_ROWS = 4
SCREENING_CANDIDATE_FREQUENCIES = 15
FINAL_CONFIG_FROZEN = TRUE
FINAL_TEST_COMPLETE = TRUE
TRAINING_PERFORMED = FALSE
SCIENTIFIC_TRAINING_RUN = FALSE
TEST_DATA_USED_FOR_SCREENING = FALSE
FINAL_TEST_RERUN = FALSE
TEST_EVALUATION_COUNT = 1
TEST_TUNING = FALSE


## Frozen scope

Use only the manifest's frozen checkpoint, source-model configuration, architecture, and preprocessing for the manuscript's existing frequency-screening demonstration. The workflow is inference-only: no training, checkpoint selection, lambda tuning, test-set tuning, or modification of Notebook 34 results is permitted.

The notebook is saved with `RUN_FREQUENCY_SCREENING = False`, which performs the complete fail-closed provenance and input preflight without inference or report creation. Changing the flag to `True` is the sole explicit authorization gate for the inference-only demonstration and its evidence reports.